In [ ]:
import numpy as np
import pandas as pd
import random
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, make_scorer
from sklearn.impute import KNNImputer
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.inspection import permutation_importance
from sklearn.utils import resample
from sklearn.model_selection import StratifiedKFold

import os
import requests

from tensorflow.keras.models import load_model

In [ ]:
def download_if_needed(url, local_path):
    """Download a file from GitHub if it doesn't already exist."""
    if os.path.exists(local_path):
        print(f"✓ Using existing {local_path}")
        return

    print(f"Downloading {local_path}...")
    response = requests.get(url)
    response.raise_for_status()

    with open(local_path, "wb") as f:
        f.write(response.content)

    print("Download complete.")

In [ ]:
HH_MODEL_URL = ("https://raw.githubusercontent.com/Earlyrizer64/MyST_site/main/Reference_Files/Chemical_Property_Database/model_v5_HH_okayvaltest.h5")
CC_MODEL_URL = ("https://raw.githubusercontent.com/Earlyrizer64/MyST_site/main/Reference_Files/Chemical_Property_Database/model_v5_CC_okayvaltest.h5")

HH_MODEL_PATH = "model_v5_HH_okayvaltest.h5"
CC_MODEL_PATH = "model_v5_CC_okayvaltest.h5"

DATABASE_PATH = "https://raw.githubusercontent.com/Earlyrizer64/MyST_site/main/Reference_Files/Chemical_Property_Database/Processed_Solvent_DF_v6_TEST.xlsx"


download_if_needed(HH_MODEL_URL, HH_MODEL_PATH)
download_if_needed(CC_MODEL_URL, CC_MODEL_PATH)

HH_MODEL = HH_MODEL_PATH
CC_MODEL = CC_MODEL_PATH

In [ ]:
model_CC = tf.keras.models.load_model(
    CC_MODEL_PATH,
    custom_objects={"LeakyReLU": tf.keras.layers.LeakyReLU},
    compile=False
)

model_HH = tf.keras.models.load_model(
    HH_MODEL_PATH,
    custom_objects={"LeakyReLU": tf.keras.layers.LeakyReLU},
    compile=False
)


In [ ]:
# 1. Example: Ethanol
MOLECULE = "PHENOL"
SMILES = "Oc1ccccc1"

# 2. Read the DB:
db = pd.read_excel(DATABASE_PATH)
db = db.set_index('SMILES')
descriptors = db.loc[SMILES]

# Selected properties for Human Health Impact:
thermo_feat_HH   = ['Heat of Vaporization(J/mol)', 'Heat Capacity (kJ/kgC)', 'XLogP','Pitzer’s Acentric Factor [-]', 'Critical Temperature [K]']
mol_desc_feat_HH = ['Chi0n', 'HallKierAlpha', 'SMR_VSA7', 'VSA_EState6','NumValenceElectrons']

# Selected properties for Climate Change:
thermo_feat_CC   = ['Heat Capacity (kJ/kgC)', 'Boiling Point(K)', 'XLogP', 'Critical Temperature [K]', 'Critical Molar Volume [m3/mol]']
mol_desc_feat_CC = ['BertzCT', 'ExactMolWt', 'HallKierAlpha', 'PEOE_VSA6', 'NOCount']

# 3. Obtain the relevant properties:
descriptors_hh = descriptors.loc[thermo_feat_HH + mol_desc_feat_HH]
descriptors_cc = descriptors.loc[thermo_feat_CC + mol_desc_feat_CC]

In [ ]:
descriptors_cc

In [ ]:
molecule_climate_change_impact = model_CC.predict(np.array([list(descriptors_cc)]), verbose=0)  # kgCO2-eq/kg chemcal
molecule_human_health_impact = model_HH.predict(np.array([list(descriptors_hh)]), verbose=0) # DALY/kg chemical

In [ ]:
print("Climate Change Impact: ", round(molecule_climate_change_impact.item(), 4), f"kgCO2-eq/kg {MOLECULE}")
print("Human Health Impact: ", round(molecule_human_health_impact.item(), 4), f"DALY/kg {MOLECULE}")

In [ ]:
import ipywidgets as widgets
from IPython.display import display

kg_slider = widgets.FloatSlider(value=10, min=0, max=100, step=0.1, description='kg amount:')
output = widgets.Output()

def update(change):
    with output:
        output.clear_output()
        kg = kg_slider.value
        total_cc = molecule_climate_change_impact[0][0] * kg
        total_hh = molecule_human_health_impact[0][0] * kg
        print(f"Climate impact: {total_cc:.4f} kgCO2-eq")
        print(f"Health impact:  {total_hh:.6f} Disability-Adjusted Life Years (DALY)")

kg_slider.observe(update, names='value')
display(kg_slider, output)
update(None)

In [ ]:
import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np

kg_slider = widgets.FloatSlider(value=10, min=0, max=100, step=0.1, description='kg amount:')
output = widgets.Output()

cc_per_kg = molecule_climate_change_impact[0][0]
hh_per_kg = molecule_human_health_impact[0][0]
kg_range = np.linspace(0, 100, 200)

def update(change):
    with output:
        output.clear_output(wait=True)
        kg = kg_slider.value
        total_cc = cc_per_kg * kg
        total_hh = hh_per_kg * kg

        fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

        axes[0].plot(kg_range, cc_per_kg * kg_range, color='#e0a458')
        axes[0].scatter([kg], [total_cc], color='#e0a458', zorder=5, s=80)
        axes[0].set_title(f"Climate: {total_cc:.3f} kgCO2-eq")
        axes[0].set_xlabel("kg chemical")

        axes[1].plot(kg_range, hh_per_kg * kg_range, color='#e0685c')
        axes[1].scatter([kg], [total_hh], color='#e0685c', zorder=5, s=80)
        axes[1].set_title(f"Health: {total_hh:.6f} Disability-Adjusted Life Years (DALY)")
        axes[1].set_xlabel("kg chemical")

        plt.tight_layout()
        plt.show()

kg_slider.observe(update, names='value')
display(kg_slider, output)
update(None)